## Imports

In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [34]:
data = pd.read_csv('../M1_final.csv')
data.head()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,TAIL_NUM,DEST,DEP_DELAY,CRS_ELAPSED_TIME,DISTANCE,CRS_DEP_M,...,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Condition,sch_dep,sch_arr,TAXI_OUT
0,11,1,5,B6,N828JB,CHS,-1,124,636,324,...,34,58,W,25,38,29.86,Fair / Windy,9,17,14
1,11,1,5,B6,N992JB,LAX,-7,371,2475,340,...,34,58,W,25,38,29.86,Fair / Windy,9,17,15
2,11,1,5,B6,N959JB,FLL,40,181,1069,301,...,34,58,W,25,38,29.86,Fair / Windy,9,17,22
3,11,1,5,B6,N999JQ,MCO,-2,168,944,345,...,34,58,W,25,38,29.86,Fair / Windy,9,17,12
4,11,1,5,DL,N880DN,ATL,-4,139,760,360,...,32,58,W,24,35,29.91,Fair / Windy,9,17,13


## Data Preprocessing

In [35]:
# Drop unnecessary features
data.drop(columns=['TAIL_NUM', 'DEP_TIME_M', 'TAXI_OUT'], inplace=True)

# create target variable
data['is_delayed'] = np.where(
    data['DEP_DELAY'] >= 15,
    1,
    0
)
data.drop(columns=['DEP_DELAY'], inplace=True)

# Drop null rows
data.dropna(inplace=True)

## Stratified train test split

In [36]:
# Stratified Train test split
from sklearn.model_selection import StratifiedShuffleSplit

x = data.drop('is_delayed', axis=1)
y = data['is_delayed']

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in splitter.split(data, data['is_delayed']):
    x_train = x.iloc[train_index]
    x_test = x.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

## Transformers

In [37]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

### Wind Transformer

In [38]:
from sklearn.base import BaseEstimator, TransformerMixin

class WindDirectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Wind'):
        self.column = column
        self.wind_dict = {
            'NNW': 340, 'CALM': 0, 'NNE': 20, 'NE': 45, 'VAR': 0, 'WSW': 230, 
            'S': 180, 'SSW': 200, 'WNW': 290, 'ESE': 115, 'N': 360, 'SW': 225, 
            'E': 90, 'W': 270, 'SSE': 155, 'ENE': 70, 'NW': 315, 'SE': 135
        }

    def fit(self, X, y=None):
        return self
    

    def transform(self, X):
        X = X.copy()
        # Map wind directions to degrees
        X['wind_deg'] = X[self.column].map(self.wind_dict)
        # Convert to radians
        X['wind_rad'] = np.deg2rad(X['wind_deg'])
        #Compute sin and cos
        X['wind_sin'] = np.sin(X['wind_rad'])
        X['wind_cos'] = np.cos(X['wind_rad'])
        # Drop original columns
        X = X.drop(columns=[self.column, 'wind_deg', 'wind_rad'])
        return X

### Dew Point Transformer

In [39]:
class DewPointTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Dew Point'):
        self.column = column
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Clean column
        X[self.column] = (
            X[self.column].astype(str).str.replace('\xa0', '', regex=False).str.strip()
        )
        # Convert them into numeric values
        X[self.column] = pd.to_numeric(X[self.column], errors='coerce')

        return X

## Create Pipeline

In [40]:
# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('Wind transformer', WindDirectionTransformer(column='Wind'), ['Wind']),
    ('Dew Point Transformer', DewPointTransformer(column='Dew Point'), ['Dew Point']),
    ('Categorical', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['DEST', 'OP_UNIQUE_CARRIER', 'Condition'])
], remainder='passthrough')

In [41]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

## Create Pipeline

In [42]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('Preprocessor', preprocessor),
    ('RandomForest', rf)
])

## Train the model

In [43]:
pipe.fit(X=x_train, y=y_train)

,steps,"[('Preprocessor', ...), ('RandomForest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Wind transformer', ...), ('Dew Point Transformer', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Check the accuracy

In [44]:
y_pred = pipe.predict(x_test)

In [45]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.8861901457321305

In [46]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, x_train, y_train, cv=10, scoring='accuracy').mean() * 100

np.float64(88.19293816188271)

## Apply GridSearchCV

In [47]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

pipe_rf = Pipeline([
    ('PreProcessing', preprocessor), # Your existing preprocessor
    ('Train RF', RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [51]:
param_grid_rf = {
    'Train RF__n_estimators': [100, 200, 300],
    'Train RF__max_depth': [5, 7, 10, None],
    'Train RF__min_samples_split': [ 10, 20, 40],
    'Train RF__min_samples_leaf': [ 10, 15, 25, 50],
    'Train RF__max_features': [None, 'sqrt', 0.3], # Using 'sqrt' and 30%
    'Train RF__class_weight': ['balanced', 'balanced_subsample']
}

In [52]:
# 3. Define the cross-validation strategy (best practice)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# 4. Instantiate and run GridSearchCV
# WARNING: This search space is very large. It will take a significant amount of time.
# Total fits = 108 combinations * 5 folds = 540 fits.
print("Starting exhaustive grid search for Random Forest...")
grid_search_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv_strategy,
    scoring='accuracy', # We will evaluate with more metrics later
    n_jobs=-1,        # Use all available CPU cores
    verbose=2         # Show progress
)

grid_search_rf.fit(x_train, y_train)

Starting exhaustive grid search for Random Forest...
Fitting 5 folds for each of 864 candidates, totalling 4320 fits


[CV] END Train RF__class_weight=balanced, Train RF__max_depth=5, Train RF__max_features=None, Train RF__min_samples_leaf=10, Train RF__min_samples_split=10, Train RF__n_estimators=100; total time=   6.2s
[CV] END Train RF__class_weight=balanced, Train RF__max_depth=5, Train RF__max_features=None, Train RF__min_samples_leaf=10, Train RF__min_samples_split=10, Train RF__n_estimators=100; total time=   6.4s
[CV] END Train RF__class_weight=balanced, Train RF__max_depth=5, Train RF__max_features=None, Train RF__min_samples_leaf=10, Train RF__min_samples_split=10, Train RF__n_estimators=100; total time=   7.1s
[CV] END Train RF__class_weight=balanced, Train RF__max_depth=5, Train RF__max_features=None, Train RF__min_samples_leaf=10, Train RF__min_samples_split=10, Train RF__n_estimators=100; total time=   7.5s
[CV] END Train RF__class_weight=balanced, Train RF__max_depth=5, Train RF__max_features=None, Train RF__min_samples_leaf=10, Train RF__min_samples_split=10, Train RF__n_estimators=100;

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'Train RF__class_weight': ['balanced', 'balanced_subsample'], 'Train RF__max_depth': [5, 7, ...], 'Train RF__max_features': [None, 'sqrt', ...], 'Train RF__min_samples_leaf': [10, 15, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Wind transformer', ...), ('Dew Point Transformer', ...), ...]"


In [54]:
grid_search_rf.best_params_

{'Train RF__class_weight': 'balanced',
 'Train RF__max_depth': None,
 'Train RF__max_features': None,
 'Train RF__min_samples_leaf': 10,
 'Train RF__min_samples_split': 10,
 'Train RF__n_estimators': 200}

In [55]:
grid_search_rf.best_score_

np.float64(0.8745115965735055)

## Randomized search CV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Use the exact same large grid you defined
param_grid_rf = {
    'Train RF__n_estimators': [10, 50, 100, 150, 200, 300],
    'Train RF__max_depth': [3, 5, 7, 10, 15, 20, None],
    'Train RF__min_samples_split': [2, 5, 10, 20, 40],
    'Train RF__min_samples_leaf': [1, 2, 4, 10, 15, 20, 25, 50],
    'Train RF__max_features': [None, 'sqrt', 0.3, 'log2'],
    'Train RF__class_weight': [None, 'balanced', 'balanced_subsample']
}

In [ ]:
# The pipeline and cv_strategy are the same as before
# ...

# Instantiate RandomizedSearchCV
# n_iter=100 means we will try 100 random combinations from your grid.
# This is a huge reduction from the 20,160 combinations in GridSearchCV.
random_search_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_grid_rf,
    n_iter=100, # <<<<< KEY PARAMETER: How many combinations to try
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42 # for reproducibility
)

In [ ]:
print("Starting Randomized Search...")
random_search_rf.fit(x_train, y_train)

Starting Randomized Search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END Train RF__class_weight=None, Train RF__max_depth=3, Train RF__max_features=log2, Train RF__min_samples_leaf=15, Train RF__min_samples_split=20, Train RF__n_estimators=100; total time=   1.4s
[CV] END Train RF__class_weight=None, Train RF__max_depth=3, Train RF__max_features=log2, Train RF__min_samples_leaf=15, Train RF__min_samples_split=20, Train RF__n_estimators=100; total time=   1.8s
[CV] END Train RF__class_weight=None, Train RF__max_depth=3, Train RF__max_features=log2, Train RF__min_samples_leaf=15, Train RF__min_samples_split=20, Train RF__n_estimators=100; total time=   1.8s
[CV] END Train RF__class_weight=None, Train RF__max_depth=3, Train RF__max_features=log2, Train RF__min_samples_leaf=15, Train RF__min_samples_split=20, Train RF__n_estimators=100; total time=   1.8s
[CV] END Train RF__class_weight=None, Train RF__max_depth=3, Train RF__max_features=log2, Train RF__min_sam

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'Train RF__class_weight': [None, 'balanced', ...], 'Train RF__max_depth': [3, 5, ...], 'Train RF__max_features': [None, 'sqrt', ...], 'Train RF__min_samples_leaf': [1, 2, ...], ...}"
,n_iter,100
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [56]:
# --- Check the results ---
print("\nRandomized Search complete.")
print(f"Best Hyperparameters found: {random_search_rf.best_params_}")
random_search_rf.best_params_


Randomized Search complete.
Best Hyperparameters found: {'Train RF__n_estimators': 100, 'Train RF__min_samples_split': 10, 'Train RF__min_samples_leaf': 1, 'Train RF__max_features': None, 'Train RF__max_depth': 20, 'Train RF__class_weight': 'balanced'}


{'Train RF__n_estimators': 100,
 'Train RF__min_samples_split': 10,
 'Train RF__min_samples_leaf': 1,
 'Train RF__max_features': None,
 'Train RF__max_depth': 20,
 'Train RF__class_weight': 'balanced'}

In [ ]:
random_search_rf.best_score_

np.float64(0.887654759367748)